# Simulated Annealing Parameter Analysis

This notebook studies how the cooling schedule affects runtime and solution quality on QUBO instances.


In [1]:
import time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt 
import os
import sys

current_dir = os.getcwd() 

root_path = os.path.abspath(os.path.join(current_dir, '..'))

if root_path not in sys.path:
    sys.path.insert(0, root_path)
from problem.qubo_problem import QuboProblem
from solver.classical_solver.simulated_annealing import SimulatedAnnealing 


N = 200 
random_matrix = sp.random(N, N, density=0.2, format='csr')

random_matrix.data -= 0.5 
Q_data = sp.triu(random_matrix, format='csr')
qubo = QuboProblem(Q_data)


In [2]:
def run_experiment(initial_temp, decreasing_factor, steps_per_temp, runs=5):
    costs = []
    durations = []

    for _ in range(runs):
        sa = SimulatedAnnealing(
            initial_temp=initial_temp, 
            decreasing_factor=decreasing_factor, 
            steps_per_temp=steps_per_temp
        )
        
        start = time.time()
        _, cost = sa.solve_efficient(qubo)
        end = time.time()
        
        costs.append(cost)
        durations.append(end - start)

    return {
        "mean_cost": np.mean(costs),
        "std_cost": np.std(costs),
        "mean_time": np.mean(durations)
    }


In [3]:
configs = [
    {"it": 100, "df": 0.8, "spt": 10},    
    {"it": 100, "df": 0.95, "spt": 50},   
    {"it": 1000, "df": 0.99, "spt": 100}, 
]

results = []
for cfg in configs:
    res = run_experiment(cfg["it"], cfg["df"], cfg["spt"])
    results.append({**cfg, **res})
    print(f"Done: {cfg}")


Done: {'it': 100, 'df': 0.8, 'spt': 10}
Done: {'it': 100, 'df': 0.95, 'spt': 50}
Done: {'it': 1000, 'df': 0.99, 'spt': 100}


In [4]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results.columns = ['Initial Temperature', 'Cooling Factor', 'Steps per Temperature', 'Average Cost', 'Standard Deviation', 'Runtime (s)']
display(df_results)


,Initial Temperature,Cooling Factor,Steps per Temperature,Average Cost,Standard Deviation,Runtime (s)
0,100,0.80,10,-79.079665,13.789755,0.800604
1,100,0.95,50,-355.153244,56.666973,6.846990
2,1000,0.99,100,-2112.107754,120.044833,34.367811


The performance depends on a trade-off between speed and quality: fast cooling (Config 0) is rapid but stays stuck at poor energy levels, while slow cooling (Config 2) reaches a much better minimum of -2656.9.  The high standard deviation confirms the algorithm is stochastic and requires more steps per temperature to converge. To optimize this, the initial temperature should accept ~80% of bad moves, followed by a decreasing factor between 0.95 and 0.99 to slowly "freeze" the solution.